# 💬 NLP Sentiment Analysis on Product Reviews
**Author:** Felipe Taha Sant'Ana | **Date:** March 2026  
**Dataset:** 6,000 product reviews across 6 categories, 3-class sentiment

---
## Table of Contents
1. [Project Overview](#1) — 2. [Data Loading](#2) — 3. [EDA](#3) — 4. [Text Analysis](#4)
5. [Feature Extraction & Modeling](#5) — 6. [Evaluation](#6) — 7. [Error Analysis](#7) — 8. [Conclusions](#8)

---
<a id="1"></a>
## 1. Project Overview
### Problem
Online retailers receive thousands of reviews daily. Manually categorizing sentiment is infeasible. We build an **NLP classification pipeline** to automatically label reviews as Positive, Negative, or Neutral.

### Approach
TF-IDF vectorization with n-grams, followed by comparison of 4 classifiers: Logistic Regression, Multinomial Naive Bayes, Linear SVC, and Random Forest.


<a id="2"></a>
## 2. Data Loading & Profiling

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from collections import Counter
import string, warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = {'primary': '#10B981', 'danger': '#EF4444', 'accent': '#F59E0B', 'secondary': '#6366F1', 'info': '#0EA5E9'}
print("Libraries loaded")

In [ ]:
df = pd.read_csv('data/product_reviews.csv', parse_dates=['ReviewDate'])
print(f"Dataset: {df.shape[0]:,} reviews x {df.shape[1]} features")
print(f"\nSentiment distribution:")
for s in ['Positive', 'Negative', 'Neutral']:
    n = len(df[df['Sentiment']==s])
    print(f"  {s:10s}: {n:>5,} ({n/len(df)*100:.1f}%)")
print(f"\nCategories: {df['Category'].nunique()}")
print(f"Avg review length: {df['WordCount'].mean():.0f} words")
df.head()

<a id="3"></a>
## 3. Exploratory Data Analysis
### 3.1 Sentiment & Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
colors3 = [COLORS['primary'], COLORS['danger'], COLORS['accent']]
df['Sentiment'].value_counts().plot(kind='pie', colors=colors3, ax=axes[0],
    autopct='%1.1f%%', textprops={'fontweight': 'bold'}, wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Sentiment Distribution', fontweight='bold')
axes[0].set_ylabel('')
for sent, color in zip(['Positive','Neutral','Negative'], colors3):
    sub = df[df['Sentiment']==sent]
    axes[1].hist(sub['Rating'], bins=np.arange(0.5,6.5,1), alpha=0.6, color=color, label=sent, edgecolor='white')
axes[1].set_title('Rating by Sentiment', fontweight='bold')
axes[1].set_xlabel('Star Rating'); axes[1].legend()
plt.tight_layout(); plt.show()

### 3.2 Sentiment by Product Category

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ct = pd.crosstab(df['Category'], df['Sentiment'], normalize='index') * 100
ct[['Positive','Neutral','Negative']].plot(kind='bar', stacked=True, color=colors3, ax=ax, edgecolor='white')
ax.set_title('Sentiment Breakdown by Category', fontweight='bold', fontsize=14)
ax.set_ylabel('Percentage (%)'); ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout(); plt.show()

### 3.3 Top Words by Sentiment

In [ ]:
def get_top_words(texts, n=15):
    words = ' '.join(texts).lower().split()
    stop = {'the','a','an','is','it','i','and','to','of','for','in','this','my','was','that',
            'with','on','but','have','had','has','its','not','are','be','from','at','by','or',
            'as','so','if','do','no','just','been','very','all','more','can','will','than',
            'about','up','out','would','what','which','their','one','were','they','there','some',
            'after','been','using','bought','purchased','product','overall'}
    words = [w.strip(string.punctuation) for w in words if w.strip(string.punctuation) not in stop and len(w)>2]
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for idx, (sent, color) in enumerate(zip(['Positive','Negative','Neutral'], colors3)):
    top = get_top_words(df[df['Sentiment']==sent]['ReviewText'].tolist(), 15)
    words, counts = zip(*top)
    axes[idx].barh(range(len(words)), counts, color=color, edgecolor='white')
    axes[idx].set_yticks(range(len(words))); axes[idx].set_yticklabels(words)
    axes[idx].invert_yaxis(); axes[idx].set_title(f'Top Words - {sent}', fontweight='bold')
plt.tight_layout(); plt.show()

<a id="4"></a>
## 4. Text Analysis
### Word Count & Review Length Statistics

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for sent, color in zip(['Positive','Neutral','Negative'], colors3):
    sub = df[df['Sentiment']==sent]
    ax.hist(sub['WordCount'], bins=25, alpha=0.5, color=color, label=sent, edgecolor='white')
ax.set_title('Review Length Distribution by Sentiment', fontweight='bold', fontsize=14)
ax.set_xlabel('Word Count'); ax.legend()
plt.tight_layout(); plt.show()

print("Word Count Statistics by Sentiment:")
print(df.groupby('Sentiment')['WordCount'].describe().round(1).to_string())

<a id="5"></a>
## 5. Feature Extraction & Model Training

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=3, max_df=0.95, sublinear_tf=True)
X = tfidf.fit_transform(df['ReviewText'])
y = df['Sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"TF-IDF features: {X.shape[1]:,}")
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Multinomial NB': MultinomialNB(alpha=0.5),
    'Linear SVC': LinearSVC(max_iter=2000, C=1.0, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=20, random_state=42, n_jobs=-1),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro')
    results[name] = {
        'model': model, 'y_pred': y_pred,
        'acc': accuracy_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred, average='macro'),
        'cv_mean': cv.mean(), 'cv_std': cv.std(),
    }
    print(f"\n{'='*50}")
    print(f"{name}: Acc={results[name]['acc']:.4f}, F1-macro={results[name]['f1']:.4f}, CV={cv.mean():.4f}+/-{cv.std():.4f}")
    print(classification_report(y_test, y_pred))

<a id="6"></a>
## 6. Model Evaluation
### 6.1 Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
mnames = list(results.keys())
for idx, (metric, title) in enumerate([('acc','Accuracy'), ('f1','F1-Macro'), ('cv_mean','CV F1-Macro')]):
    vals = [results[m][metric] for m in mnames]
    bars = axes[idx].bar(mnames, vals, color=[COLORS['primary'],COLORS['danger'],COLORS['accent'],COLORS['secondary']], edgecolor='white')
    axes[idx].set_title(title, fontweight='bold')
    axes[idx].set_ylim(min(vals)-0.03, 1.01)
    axes[idx].set_xticklabels(mnames, rotation=20, ha='right', fontsize=9)
    for bar, v in zip(bars, vals):
        axes[idx].text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.003, f'{v:.3f}', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

### 6.2 Confusion Matrix (Best Model)

In [ ]:
best_name = max(results, key=lambda k: results[k]['f1'])
best = results[best_name]
fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, best['y_pred'], labels=['Positive','Neutral','Negative'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax,
    xticklabels=['Positive','Neutral','Negative'], yticklabels=['Positive','Neutral','Negative'],
    linewidths=2, linecolor='white', annot_kws={'size': 14, 'fontweight': 'bold'})
ax.set_title(f'Confusion Matrix - {best_name}', fontweight='bold', fontsize=14)
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout(); plt.show()

### 6.3 Top TF-IDF Features per Class

In [ ]:
lr = results['Logistic Regression']['model']
feature_names = np.array(tfidf.get_feature_names_out())
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for idx, (cls, color) in enumerate(zip(lr.classes_, colors3)):
    coef = lr.coef_[idx]
    top_idx = np.argsort(coef)[-15:]
    axes[idx].barh(range(15), coef[top_idx], color=color, edgecolor='white')
    axes[idx].set_yticks(range(15)); axes[idx].set_yticklabels(feature_names[top_idx])
    axes[idx].set_title(f'Top Features - {cls}', fontweight='bold')
plt.tight_layout(); plt.show()

<a id="7"></a>
## 7. Error Analysis

In [ ]:
test_df = df.iloc[y_test.index].copy()
test_df['Predicted'] = best['y_pred']
test_df['Correct'] = test_df['Sentiment'] == test_df['Predicted']

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
err_by_cat = test_df.groupby('Category')['Correct'].mean() * 100
err_by_cat.sort_values().plot(kind='barh', color=COLORS['info'], ax=axes[0], edgecolor='white')
axes[0].set_title('Accuracy by Category', fontweight='bold')
axes[0].set_xlabel('Accuracy (%)')
for i, v in enumerate(err_by_cat.sort_values().values):
    axes[0].text(v+0.2, i, f'{v:.1f}%', va='center', fontweight='bold')

misclassified = test_df[~test_df['Correct']]
if len(misclassified) > 0:
    print(f"Total misclassified: {len(misclassified)} ({len(misclassified)/len(test_df)*100:.1f}%)")
    print(f"\nMost common confusion:")
    confusion_pairs = misclassified.groupby(['Sentiment', 'Predicted']).size().sort_values(ascending=False)
    print(confusion_pairs.head(6).to_string())
else:
    print("Very few misclassifications.")

err_by_len = test_df.copy()
err_by_len['LenBin'] = pd.cut(err_by_len['WordCount'], bins=5)
len_acc = err_by_len.groupby('LenBin', observed=True)['Correct'].mean() * 100
len_acc.plot(kind='bar', color=COLORS['secondary'], ax=axes[1], edgecolor='white')
axes[1].set_title('Accuracy by Review Length', fontweight='bold')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_xticklabels([str(x) for x in len_acc.index], rotation=30, ha='right')
plt.tight_layout(); plt.show()

<a id="8"></a>
## 8. Conclusions & Future Work

### Key Results
| Model | Accuracy | F1-Macro | CV F1 |
|:------|:---------|:---------|:------|
| **Linear SVC** | **Best** | **Best** | **Best** |
| Logistic Regression | Very Strong | Very Strong | Stable |
| Multinomial NB | Good | Good | Fast |
| Random Forest | Good | Good | Slower |

### Insights
- TF-IDF with bigrams captures sentiment-bearing phrases effectively
- Linear models outperform tree-based models on text classification
- Neutral reviews are hardest to classify (mixed vocabulary)
- Product category has minimal impact on classification accuracy

### Future Work
- **Transformer models** - Fine-tune BERT/RoBERTa for improved neutral detection
- **Aspect-based sentiment** - Extract sentiment per product attribute
- **Multi-language support** - Extend to Portuguese, Spanish via XLM-RoBERTa
- **Real-time API** - Deploy as FastAPI microservice with batch processing
- **Active learning** - Prioritize uncertain samples for human labeling

---
*Synthetic data for portfolio demonstration purposes.*
